In [24]:
import numpy as np, pandas as pd, json
from pathlib import Path
from sklearn.metrics import precision_recall_curve, average_precision_score

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
LABELS = ["toxic","severe_toxic","obscene","threat","insult","identity_hate"]

val = pd.read_parquet(ROOT/"data/processed/val.parquet")
P = np.load(ROOT/"models/distilbert_val_probs.npy")
print(P.shape, len(val))   # (~16000, 6)

(15929, 6) 15929


In [25]:
def pick(y, s, p_target=0.95, r_target=0.90):
    p, r, t = precision_recall_curve(y, s)
    p, r = p[:-1], r[:-1]
    hide = float(t[np.argmax(p >= p_target)]) if (p >= p_target).any() else 1.0
    ok = np.where(r >= r_target)[0]
    review = float(t[ok.max()]) if len(ok) else 0.0
    return min(review, hide), hide

thr, rows = {}, []
for i, l in enumerate(LABELS):
    y, s = val[l].values, P[:, i]
    rv, hd = pick(y, s)
    thr[l] = {"review": round(rv, 4), "hide": round(hd, 4)}
    hid = s >= hd
    rows.append({"label": l, "positives": int(y.sum()),
                 "T_review": rv, "T_hide": hd,
                 "hide_precision": y[hid].mean() if hid.any() else np.nan,
                 "hide_recall": (hid & (y == 1)).sum() / max(y.sum(), 1),
                 "pr_auc": average_precision_score(y, s)})
print(pd.DataFrame(rows).round(3))

           label  positives  T_review  T_hide  hide_precision  hide_recall  \
0          toxic       1517     0.445   0.950           0.951        0.652   
1   severe_toxic        156     0.494   0.970           1.000        0.013   
2        obscene        876     0.622   0.981           0.951        0.571   
3         threat         39     0.041   0.983           1.000        0.231   
4         insult        797     0.500   0.988           0.952        0.124   
5  identity_hate        146     0.078   0.988           1.000        0.055   

   pr_auc  
0   0.919  
1   0.531  
2   0.916  
3   0.634  
4   0.821  
5   0.596  


In [26]:
(ROOT/"thresholds.json").write_text(json.dumps(thr, indent=2))

372

In [27]:
import sys; sys.path.insert(0, str(ROOT))
from src.decide import decide
acts = [decide(dict(zip(LABELS, P[i])), thr, val.text.iloc[i])["action"] for i in range(len(P))]
print(pd.Series(acts).value_counts(normalize=True).round(3))

publish      0.882
auto_hide    0.065
review       0.053
Name: proportion, dtype: float64
